# Ghép dữ liệu Dryad (chim cánh cụt hoàng đế) để chạy mô hình ODE
Hai file Excel trong zip:
- **Dive depth & Hb saturation data**: mỗi chim một sheet, Hb (5–15 s/mẫu) và độ sâu (1 s/mẫu) ở hai cụm cột riêng.
- **Saturation Profile Groups**: bảng 160 lần lặn (nhóm, giờ bắt đầu/kết thúc, độ sâu tối đa).

Quy trình: (1) ghép Hb với độ sâu theo thời gian, (2) đọc bảng lần lặn, (3) gắn nhãn lần lặn cho từng mẫu Hb, (4) xuất file.

In [ ]:
import datetime as dt
import numpy as np, pandas as pd, openpyxl

DIR = "../data"      # thư mục đã giải nén, chứa 2 file .xlsx
F_HB   = f"{DIR}/Dive_depth___Hb_saturation_data_2-21-24.xlsx"
F_DIVE = f"{DIR}/Saturation_Profile_Groups_-_ID___Dive_Parameters.2-21-24xlsx.xlsx"
EPOCH = dt.datetime(1970, 1, 1)

def norm_bird(s):                       # sửa lỗi gõ: 'EPO7' -> 'EP07', bỏ khoảng trắng
    return str(s).strip().replace("EPO", "EP0")

## 1. Ghép Hb với độ sâu (theo từng chim)

In [ ]:
def tsec(v, mode):
    """Đổi mọi kiểu thời gian về giây. mode='abs': giây tuyệt đối; 'tod': giây trong ngày."""
    if isinstance(v, dt.datetime):
        return (v - EPOCH).total_seconds() if mode == "abs" else v.hour*3600 + v.minute*60 + v.second
    if isinstance(v, dt.time):      return v.hour*3600 + v.minute*60 + v.second
    if isinstance(v, dt.timedelta): return v.total_seconds()
    return None

def load_sheet(ws):
    rows = list(ws.iter_rows(values_only=True))
    h = next(i for i, r in enumerate(rows) if r[0] in ("Date/Time", "Time"))
    hdr = [str(v).strip().lower() if v else None for v in rows[h]]
    data = rows[h+1:]
    po = [j for j, v in enumerate(hdr) if v == "po2"]       # có thể có 2 khối (2 ngày)
    dp = [j for j, v in enumerate(hdr) if v == "depth"]
    first = next(r[dp[0]-1] for r in data if r[dp[0]-1] is not None)
    mode = "abs" if isinstance(first, dt.datetime) else "tod"   # độ sâu chỉ có giờ -> so theo giờ trong ngày
    sat, dep = [], []
    for j in po:
        for r in data:
            if j+1 < len(r) and r[j-1] is not None and r[j] is not None and r[j+1] is not None:
                sat.append((tsec(r[j-1], mode), float(r[j]), float(r[j+1])))
    for j in dp:
        for r in data:
            if j < len(r) and r[j-1] is not None and r[j] is not None:
                dep.append((tsec(r[j-1], mode), abs(float(r[j]))))   # |độ sâu| (EP37 ghi số âm)
    s = pd.DataFrame(sat, columns=["t", "PO2", "Hb_pct"]).sort_values("t").drop_duplicates("t")
    d = pd.DataFrame(dep, columns=["t", "depth_m"]).sort_values("t").drop_duplicates("t")
    return s, d

wb = openpyxl.load_workbook(F_HB, read_only=True)
parts = []
for ws in wb.worksheets:
    s, d = load_sheet(ws)
    s["depth_m"] = np.interp(s.t, d.t, d.depth_m, left=np.nan, right=np.nan)  # nội suy độ sâu lên thời điểm đo Hb
    k = np.searchsorted(d.t.values, s.t.values).clip(1, len(d)-1)
    gap = np.minimum(abs(d.t.values[k] - s.t.values), abs(d.t.values[k-1] - s.t.values))
    s.loc[gap > 3, "depth_m"] = np.nan                       # bỏ điểm cách mẫu độ sâu > 3 s
    s["bird"] = norm_bird(ws.title.split()[0])
    s["site"] = "arterial" if "arterial" in ws.title.lower() else "venous"
    parts.append(s)
sat = pd.concat(parts, ignore_index=True)
sat["tod"] = sat.t % 86400          # giây trong ngày, dùng để so với bảng lần lặn
sat.shape

(29511, 7)

## 2. Đọc bảng lần lặn (ba khối cột Type A / Type B / Arterial -> một bảng)

In [ ]:
def sod(t): return t.hour*3600 + t.minute*60 + t.second

ws = openpyxl.load_workbook(F_DIVE).active
L = []
for r in ws.iter_rows(min_row=5, values_only=True):
    if r[0]:  L.append(("Type A venous", r[0],  r[1],  sod(r[4]),  sod(r[5]),  r[3]))
    if r[7]:  L.append(("Type B venous", r[7],  r[8],  sod(r[11]), sod(r[12]), r[10]))
    if r[14]: L.append(("Arterial",      r[14], r[15], sod(r[16]), sod(r[17]), r[19]))   # khối arterial thứ tự cột khác
D = pd.DataFrame(L, columns=["group", "bird", "dive_id", "start", "end", "max_depth"])
D["bird"] = D.bird.map(norm_bird)
D["dive_id"] = D.dive_id.astype(str).str.strip()
D["dur_min"] = (D.end - D.start) / 60
D.groupby("group").size()      # kỳ vọng: Arterial 64, Type A 31, Type B 65

group
Arterial         64
Type A venous    31
Type B venous    65
dtype: int64

## 3. Gắn nhãn lần lặn cho từng mẫu Hb

In [ ]:
sat["dive_id"] = None; sat["group"] = None
sat["t_in_dive_s"] = np.nan; sat["dive_dur_min"] = np.nan
for _, d in D.iterrows():
    m = (sat.bird == d.bird) & (sat.tod >= d.start) & (sat.tod <= d.end)
    sat.loc[m, ["dive_id", "group"]] = [d.dive_id, d.group]
    sat.loc[m, "t_in_dive_s"] = sat.loc[m, "tod"] - d.start
    sat.loc[m, "dive_dur_min"] = d.dur_min
sat["Hb"] = sat.Hb_pct / 100          # phân số 0–1, đúng đơn vị mô hình
sat = sat.sort_values(["bird", "t"]).reset_index(drop=True)

# kiểm tra nhanh
g = sat.dropna(subset=["dive_id"]).groupby("dive_id")
ok = (g.size() >= 8) & (g.dive_dur_min.first() * 60 >= 100)
print("lần lặn gắn được mẫu Hb:", g.ngroups, "/", len(D))
print("qua lọc (>=100 s, >=8 mẫu Hb):", ok.sum())
print("cặp (PO2, Hb) khác nhau:", sat[["PO2", "Hb_pct"]].drop_duplicates().shape[0])

lần lặn gắn được mẫu Hb: 153 / 160
qua lọc (>=100 s, >=8 mẫu Hb): 152
cặp (PO2, Hb) khác nhau: 260


## 4. Xuất file

In [ ]:
cols = ["bird", "site", "group", "dive_id", "t", "tod", "t_in_dive_s", "dive_dur_min", "PO2", "Hb_pct", "Hb", "depth_m"]
sat[cols].to_csv("../data/penguin_merged.csv", index=False)
with pd.ExcelWriter("../data/penguin_merged.xlsx") as w:
    sat[cols].to_excel(w, sheet_name="Hb_depth_merged", index=False)
    D.to_excel(w, sheet_name="dive_table", index=False)